# Performance Test - Noise Robustness

Notebook version of the Airflow `performance_test_dag`.

**What it does:**
1. Loads the latest registered model from MLflow (`http://localhost:5000`)
2. Loads test data from PostgreSQL (`localhost:5441`)
3. Injects increasing Gaussian noise into features
4. Measures R², RMSE, MAPE degradation
5. Plots the noise vs performance curves

**Run this from the host machine** (not inside a Docker container).

In [35]:
# ------------------- Setup ------------------- #

import numpy as np
import pandas as pd
import psycopg2
import mlflow
from mlflow.pyfunc import load_model
from mlflow.tracking import MlflowClient
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")
%matplotlib inline

In [36]:
# ------------------- Localhost config ------------------- #

MLFLOW_URL = "http://localhost:5000"
MODEL_NAME = "MODEL_EDF"
TARGET = "consommation"

DB_CONFIG = {
    "host": "localhost",
    "database": "postgres",
    "user": "postgres",
    "password": "postgres",
    "port": 5441,
}

mlflow.set_tracking_uri(MLFLOW_URL)
print(f"MLflow tracking URI: {MLFLOW_URL}")
print(f"Model name: {MODEL_NAME}")
print(f"PostgreSQL: {DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")

MLflow tracking URI: http://localhost:5000
Model name: MODEL_EDF
PostgreSQL: localhost:5441/postgres


## 1. Load test data from PostgreSQL

In [37]:
# ------------------- Load test data ------------------- #

FEATURE_COLUMNS = [
    "temp_fr", "snow_fr", "hour", "month", "dayofweek", "weekend",
    "consommation",
]

SOURCE_TABLE = "agg_conso_meteo_features"
FALLBACK_TABLE = "aggregated_conso_weather"

test_years = [2020]
test_months = [1, 2, 3]

def load_data(years, months, days=None):
    year_ph = ",".join(["%s"] * len(years))
    month_ph = ",".join(["%s"] * len(months))
    params = list(years) + list(months)
    day_filter = ""
    if days:
        day_ph = ",".join(["%s"] * len(days))
        day_filter = f" AND day IN ({day_ph})"
        params.extend(days)

    for table in [SOURCE_TABLE, FALLBACK_TABLE]:
        try:
            conn = psycopg2.connect(**DB_CONFIG)
            query = f"""
                SELECT * FROM {table}
                WHERE year IN ({year_ph})
                  AND month IN ({month_ph})
                  {day_filter}
                ORDER BY datetime
            """
            df = pd.read_sql(query, conn, params=tuple(params))
            conn.close()
            if len(df) > 0:
                print(f"Loaded {len(df)} rows from {table}")
                return df
        except Exception as e:
            print(f"{table}: {e}")

    raise ValueError(f"No data found for years={years}, months={months}")

df = load_data(test_years, test_months)
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

X = df.select_dtypes(include=["number"]).drop(columns=[TARGET], errors="ignore").fillna(0)
y = df[TARGET].fillna(0)

# Use a fixed sample for reproducibility
X_test = X.head(1000)
y_test = y.head(1000)

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"Features: {list(X_test.columns)}")

Loaded 2184 rows from agg_conso_meteo_features
X_test shape: (1000, 10)
y_test shape: (1000,)
Features: ['temp_fr', 'snow_fr', 'hour', 'month', 'dayofweek', 'weekend', 'id', 'conso_id', 'year', 'day']


## 2. Train XGBoost model and push to MLflow

In [38]:
# ------------------- Train XGBoost model and push to MLflow ------------------- #

from xgboost import XGBRegressor
from datetime import datetime
import mlflow
import mlflow.sklearn

# Load a broader training set (different years than test: 2017-2019)
train_years = [2017, 2018, 2019]
train_months = list(range(1, 13))

df_train = load_data(train_years, train_months)
df_train[TARGET] = pd.to_numeric(df_train[TARGET], errors="coerce")

# Select only numeric columns (XGBoost can't handle datetime64)
# Drop the target + non-feature columns (id, conso_id, datetime)
non_feature_cols = {TARGET, "id", "conso_id", "datetime"}
feature_cols = [
    c for c in df_train.select_dtypes(include=["number"]).columns
    if c not in non_feature_cols
]
X_all = df_train[feature_cols].fillna(0)
y_all = pd.to_numeric(df_train[TARGET], errors="coerce").fillna(0)

print(f"Training data: {len(X_all)} samples, {len(feature_cols)} features")
print(f"Features: {feature_cols}")

# Train XGBoost
model_type = "XGBoost"
xgb = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb.fit(X_all, y_all)
print(f"XGBoost trained on {len(X_all)} samples")

# Log to MLflow
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_name = f"{MODEL_NAME}_{model_type}_{timestamp}"
mlflow.set_tracking_uri(MLFLOW_URL)
experiment_id = mlflow.create_experiment(experiment_name)

with mlflow.start_run(experiment_id=experiment_id, run_name=f"training_{model_type}_{timestamp}") as run:
    mlflow.log_param("model_type", model_type)
    mlflow.log_param("selected_years", str(train_years))
    mlflow.log_param("selected_months", str(train_months))
    mlflow.log_param("features", str(feature_cols))
    mlflow.log_param("n_samples", len(X_all))
    mlflow.log_param("n_features", len(feature_cols))
    mlflow.sklearn.log_model(xgb, artifact_path=MODEL_NAME)
    print(f"Model logged to experiment '{experiment_name}', run {run.info.run_id}")

mlflow.end_run()


Loaded 26280 rows from agg_conso_meteo_features
Training data: 26280 samples, 8 features
Features: ['temp_fr', 'snow_fr', 'hour', 'month', 'dayofweek', 'weekend', 'year', 'day']
XGBoost trained on 26280 samples


2026/07/06 23:49:51 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Model logged to experiment 'MODEL_EDF_XGBoost_20260706_234944', run 83390af57eda4c879f361e257204dd91
🏃 View run training_XGBoost_20260706_234944 at: http://localhost:5000/#/experiments/6/runs/83390af57eda4c879f361e257204dd91
🧪 View experiment at: http://localhost:5000/#/experiments/6


## 3. Load model from MLflow

In [39]:
# ------------------- Load model from MLflow (interactive: pick experiment + run) ------------------- #

import sys
sys.path.insert(0, "dags")

from datetime import datetime
from mlflow_utils import get_client as _get_client
from mlflow_utils import resolve_model_artifact, load_model_from_run

client = _get_client(MLFLOW_URL)

# --- 1. List all experiments ---
experiments = client.search_experiments(
    view_type=mlflow.entities.ViewType.ACTIVE_ONLY,
    order_by=["creation_time DESC"],
)
print("Available experiments:")
for exp in experiments:
    print(f"  [{exp.experiment_id}] {exp.name}")

# --- 2. User picks experiment (ID or name) ---
raw = input("\nEnter experiment ID or name: ").strip()
exp = client.get_experiment(raw) if raw.isdigit() else client.get_experiment_by_name(raw)
if not exp:
    raise ValueError(f"Experiment {raw!r} not found")
print(f"\nSelected: [{exp.experiment_id}] {exp.name}")

# --- 3. List runs in that experiment ---
runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["attribute.start_time DESC"],
)
if not runs:
    raise ValueError(f"Experiment {exp.experiment_id!r} has no runs")

print(f"\nRuns in '{exp.name}' ({len(runs)} total):")
for i, r in enumerate(runs):
    run_name = r.data.tags.get("mlflow.runName", "unnamed")
    ts = datetime.fromtimestamp(r.info.start_time / 1000).strftime("%Y-%m-%d %H:%M:%S") if r.info.start_time else "?"
    print(f"  [{i}] {r.info.run_id[:12]}...  {run_name}  ({ts})")

# --- 4. User picks a run ---
run_idx = input(f"\nSelect run index [0-{len(runs)-1}] (default: 0): ").strip()
if not run_idx:
    run_idx = 0
else:
    run_idx = int(run_idx)
run = runs[run_idx]
print(f"Selected run: {run.info.run_id}")

# --- 5. Resolve artifact path and load model ---
artifact_path = resolve_model_artifact(run, use_fallback=False)
print(f"Artifact path: {artifact_path}")
model = load_model_from_run(run.info.run_id, artifact_path)

# --- 6. Expose variables expected by downstream cells ---
resolved_exp = exp.name
model_version = exp.name.rsplit("_", 1)[-1] if "_" in exp.name else exp.experiment_id

print(f"\nModel loaded from experiment {exp.name!r} (ID: {exp.experiment_id}), run {run.info.run_id}")


Available experiments:
  [6] MODEL_EDF_XGBoost_20260706_234944
  [5] MODEL_EDF_XGBoost_20260706_234807
  [4] EDF_Model_Experiment
  [3] MODEL_EDF_RandomForest_20260706_160955
  [2] Performance_Testy[2020]_m[4]
  [1] MODEL_EDF_KNN_20260705_155658
  [0] Default

Selected: [3] MODEL_EDF_RandomForest_20260706_160955

Runs in 'MODEL_EDF_RandomForest_20260706_160955' (1 total):
  [0] eec75f36e035...  training_RandomForest_20260706_160955  (2026-07-06 18:09:55)
Selected run: eec75f36e035469b8f1adef1ec63948d
Artifact path: MODEL_EDF


OSError: No such file or directory: './mlflow/artifacts/3/eec75f36e035469b8f1adef1ec63948d/artifacts/MODEL_EDF'

## 4. Noise injection test

In [ ]:
# ------------------- Noise injection ------------------- #

from sklearn.metrics import r2_score, mean_squared_error

def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

def add_noise(X, noise_level, seed=42):
    np.random.seed(seed)
    X_noisy = X.copy()
    for col in X_noisy.select_dtypes(include=[np.number]).columns:
        col_std = X_noisy[col].std()
        if col_std > 0:
            noise = np.random.normal(0, noise_level * col_std, size=X_noisy[col].shape)
            X_noisy[col] = X_noisy[col] + noise
    return X_noisy

noise_levels = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]

metrics = {"R2": [], "RMSE": [], "MAPE (%)": []}
predictions = []

print(f"{'Noise':<10} {'R2':<12} {'RMSE':<12} {'MAPE (%)':<12}")
print("-" * 50)

for i, nl in enumerate(noise_levels):
    X_n = add_noise(X_test, nl, seed=i) if nl > 0 else X_test.copy()
    y_pred = model.predict(X_n)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mp = mape(y_test, y_pred)

    metrics["R2"].append(r2)
    metrics["RMSE"].append(rmse)
    metrics["MAPE (%)"].append(mp)
    predictions.append(y_pred)

    print(f"{nl:<10.2f} {r2:<12.4f} {rmse:<12.4f} {mp:<12.2f}")

## 5. Degradation relative (en %)

In [ ]:
# ------------------- Degradation calculation ------------------- #

baseline = {"R2": metrics["R2"][0], "RMSE": metrics["RMSE"][0], "MAPE (%)": metrics["MAPE (%)"][0]}

degradation = {"R2": [], "RMSE": [], "MAPE (%)": []}

print(f"Baseline (noise=0.0): R2={baseline['R2']:.4f}, RMSE={baseline['RMSE']:.4f}, MAPE={baseline['MAPE (%)']:.2f}%")
print()
print(f"{'Noise':<10} {'R2 Deg (%)':<15} {'RMSE Deg (%)':<15} {'MAPE Deg (%)':<15}")
print("-" * 60)

for i, nl in enumerate(noise_levels):
    if nl > 0:
        r2d = ((metrics["R2"][i] - baseline["R2"]) / abs(baseline["R2"])) * 100
        rmsed = ((metrics["RMSE"][i] - baseline["RMSE"]) / baseline["RMSE"]) * 100
        maped = ((metrics["MAPE (%)"][i] - baseline["MAPE (%)"]) / baseline["MAPE (%)"]) * 100
    else:
        r2d = rmsed = maped = 0.0

    degradation["R2"].append(r2d)
    degradation["RMSE"].append(rmsed)
    degradation["MAPE (%)"].append(maped)

    print(f"{nl:<10.2f} {r2d:<15.2f} {rmsed:<15.2f} {maped:<15.2f}")

## 6. Visualisation

In [ ]:
# ------------------- Plot: raw metrics ------------------- #

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle(f'Model Performance vs Noise Level (v{model_version})', fontsize=14, fontweight='bold')

axes[0].plot(noise_levels, metrics["R2"], 'o-', linewidth=2, markersize=8)
axes[0].set_xlabel('Noise Level')
axes[0].set_ylabel('R² Score')
axes[0].set_title('R²')
axes[0].grid(True, alpha=0.3)

axes[1].plot(noise_levels, metrics["RMSE"], 's-', color='orange', linewidth=2, markersize=8)
axes[1].set_xlabel('Noise Level')
axes[1].set_ylabel('RMSE')
axes[1].set_title('RMSE')
axes[1].grid(True, alpha=0.3)

axes[2].plot(noise_levels, metrics["MAPE (%)"], '^-', color='green', linewidth=2, markersize=8)
axes[2].set_xlabel('Noise Level')
axes[2].set_ylabel('MAPE (%)')
axes[2].set_title('MAPE')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ------------------- Plot: degradation ------------------- #

fig, ax = plt.subplots(figsize=(10, 5.5))

ax.plot(noise_levels, degradation["R2"], 'o-', label='R² Degradation', linewidth=2, markersize=8, color='red')
ax.plot(noise_levels, degradation["RMSE"], 's-', label='RMSE Degradation', linewidth=2, markersize=8, color='blue')
ax.plot(noise_levels, degradation["MAPE (%)"], '^-', label='MAPE Degradation', linewidth=2, markersize=8, color='green')

ax.axhline(y=0, color='black', linestyle='--', alpha=0.5, linewidth=2)
ax.axhline(y=-10, color='orange', linestyle=':', alpha=0.5, linewidth=1.5, label='-10% warning')
ax.axhline(y=-20, color='red', linestyle=':', alpha=0.5, linewidth=1.5, label='-20% critical')

ax.set_xlabel('Noise Level (fraction of std)')
ax.set_ylabel('Relative Degradation (%)')
ax.set_title(f'Metric Degradation vs Noise Level - v{model_version}', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)

for i, nl in enumerate(noise_levels):
    if degradation["R2"][i] < -20:
        ax.annotate(f'critical @ {nl:.2f}',
                    xy=(nl, degradation["R2"][i]),
                    xytext=(10, 10), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7),
                    arrowprops=dict(arrowstyle='->'))

plt.tight_layout()
plt.show()

## 7. Summary results table

In [ ]:
# ------------------- Summary table ------------------- #

summary = pd.DataFrame({
    "noise_level": noise_levels,
    "R2": [f"{v:.4f}" for v in metrics["R2"]],
    "RMSE": [f"{v:.4f}" for v in metrics["RMSE"]],
    "MAPE (%)": [f"{v:.2f}" for v in metrics["MAPE (%)"]],
    "R2 Deg (%)": [f"{v:.2f}" for v in degradation["R2"]],
    "RMSE Deg (%)": [f"{v:.2f}" for v in degradation["RMSE"]],
    "MAPE Deg (%)": [f"{v:.2f}" for v in degradation["MAPE (%)"]],
})

print(f"Model: {MODEL_NAME} v{model_version}")
print(f"Test data: years={test_years}, months={test_months}")
print(f"Baseline R²: {baseline['R2']:.4f}")
print()
summary

In [ ]:
# ------------------- Rapport de Test : Robustesse au bruit ------------------- #

nl_30_idx = noise_levels.index(0.3)
r2_at_30 = metrics["R2"][nl_30_idx]
r2_degradation_at_30 = degradation["R2"][nl_30_idx]
rmse_at_30 = metrics["RMSE"][nl_30_idx]
rmse_degradation_at_30 = degradation["RMSE"][nl_30_idx]
mape_at_30 = metrics["MAPE (%)"][nl_30_idx]
mape_degradation_at_30 = degradation["MAPE (%)"][nl_30_idx]

# Déterminer le statut
r2_ok = r2_degradation_at_30 > -20
rmse_ok = rmse_degradation_at_30 < 20  # augmentation acceptable
mape_ok = mape_degradation_at_30 < 20

if r2_ok and rmse_ok and mape_ok:
    statut = "VALIDÉ"
    emoji = "✅"
elif r2_degradation_at_30 > -50:
    statut = "VALIDÉ AVEC RÉSERVES"
    emoji = "⚠️"
else:
    statut = "NON VALIDÉ"
    emoji = "❌"

print()
print("─" * 65)
print("   RAPPORT DE TEST DE ROBUSTESSE AU BRUIT")
print("─" * 65)
print(f"   Test du Modèle {MODEL_NAME} sur les données météo+conso")
print(f"   Période : années={test_years}, mois={test_months}")
print()
print(f"   ID du Test")
print(f"   TST-REG-001")
print()
print(f"   Objectif du test")
print(f"   Vérifier la robustesse au bruit gaussien")
print()
print(f"   Description")
print(f"   Ajouter du bruit gaussien aux features météorologiques")
print()
print(f"   Données Utilisées")
print(f"   Agrégation conso + météo (temp_fr, snow_fr, hour, month, dayofweek, weekend)")
print()
print(f"   Méthode")
print(f"   Injection de bruit gaussien (moyenne=0, écart-type = noise_level × feature_std)")
print()
print(f"   Résultat attendu")
print(f"   R² ≥ 0.70, dégradation R² < 20% à noise=0.30")
print(f"   RMSE sans rupture brutale, MAPE stable")
print()
print(f"   Résultat obtenu")
print(f"   R² : {baseline['R2']:.2f} → {r2_at_30:.2f} (dégradation {r2_degradation_at_30:.1f}%)")
print(f"   RMSE : {baseline['RMSE']:.0f} → {rmse_at_30:.0f} (augmentation {rmse_degradation_at_30:.1f}%)")
print(f"   MAPE : {baseline['MAPE (%)']:.2f}% → {mape_at_30:.2f}% (dégradation {mape_degradation_at_30:.1f}%)")
print()
print(f"   Statut")
print(f"   {emoji} {statut}")
print()
print(f"   Commentaire")
print(f"   Le modèle démontre une robustesse {'satisfaisante' if r2_ok else 'limitée'} au bruit gaussien")
print(f"   jusqu’à un facteur de 0.30 avec une dégradation de {r2_degradation_at_30:.1f}% du R².")
if not r2_ok:
    print(f"   ATTENTION : la dégradation dépasse le seuil de 20% à noise=0.30.")
print("─" * 65)

In [ ]:
# ------------------- Rapport de Test : Performance du Modèle ------------------- #

from sklearn.metrics import mean_absolute_error

# Baseline predictions (noise = 0.0)
y_pred_baseline = predictions[0]

# Compute all performance metrics
r2_perf = r2_score(y_test, y_pred_baseline)
rmse_perf = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
mae_perf = mean_absolute_error(y_test, y_pred_baseline)
mape_perf = mape(y_test, y_pred_baseline)

# Acceptability thresholds
R2_THRESHOLD = 0.70
MAPE_THRESHOLD = 15.0

# Determine status
r2_ok = r2_perf >= R2_THRESHOLD
mape_ok = mape_perf <= MAPE_THRESHOLD

if r2_ok and mape_ok:
    statut = "VALIDÉ"
    emoji = "✅"
elif r2_perf >= 0.50:
    statut = "VALIDÉ AVEC RÉSERVES"
    emoji = "⚠️"
else:
    statut = "NON VALIDÉ"
    emoji = "❌"

print()
print("─" * 65)
print("   RAPPORT DE TEST DE PERFORMANCE")
print("─" * 65)
print(f"   Test du Modèle {MODEL_NAME} sur les données météo+conso")
print(f"   Période : années={test_years}, mois={test_months}")
print()
print(f"   ID du Test")
print(f"   TST-PERF-001")
print()
print(f"   Objectif du test")
print(f"   Vérifier la cohérence des métriques de performance sur le jeu de test")
print()
print(f"   Description")
print(f"   Évaluer les métriques de régression (R², RMSE, MAPE, MAE) sur les données de test propres")
print()
print(f"   Données Utilisées")
print(f"   Agrégation conso + météo (temp_fr, snow_fr, hour, month, dayofweek, weekend)")
print()
print(f"   Méthode")
print(f"   Prédiction sur le jeu de test sans bruit, calcul des métriques de performance")
print()
print(f"   Résultat attendu")
print(f"   R² ≥ {R2_THRESHOLD}, MAPE ≤ {MAPE_THRESHOLD}%")
print(f"   RMSE cohérent avec l'échelle de la consommation")
print(f"   MAE cohérent (MAE < RMSE)")
print()
print(f"   Résultat obtenu")
check_mark = chr(10003)  # ✓
cross_mark = chr(10007)  # ✗
r2_status = check_mark if r2_ok else cross_mark
rmse_status = check_mark if mae_perf < rmse_perf else cross_mark
mape_status = check_mark if mape_ok else cross_mark
print(f"   R² : {r2_perf:.4f} {r2_status} (seuil ≥ {R2_THRESHOLD})")
print(f"   RMSE : {rmse_perf:.2f}")
print(f"   MAE : {mae_perf:.2f} {rmse_status} (MAE < RMSE)")
print(f"   MAPE : {mape_perf:.2f}% {mape_status} (seuil ≤ {MAPE_THRESHOLD}%)")
print()
print(f"   Statut")
print(f"   {emoji} {statut}")
print()
print(f"   Commentaire")
if r2_ok and mape_ok:
    print(f"   Le modèle atteint un R² de {r2_perf:.4f} et un MAPE de {mape_perf:.2f}%,")
    print(f"   démontrant une performance satisfaisante sur les données de test.")
    print(f"   Les métriques RMSE ({rmse_perf:.2f}) et MAE ({mae_perf:.2f}) sont cohérentes")
    print(f"   avec l'échelle de la consommation énergétique.")
elif r2_ok:
    print(f"   Le R² de {r2_perf:.4f} est acceptable mais le MAPE ({mape_perf:.2f}%)")
    print(f"   dépasse le seuil de {MAPE_THRESHOLD}%.")
else:
    print(f"   Performance insuffisante : R²={r2_perf:.4f}, MAPE={mape_perf:.2f}%.")
    print(f"   Un ré-entraînement ou un ajustement des features est recommandé.")
print("─" * 65)
